In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import BytesIO

BASE_URL = (
    "https://www.ncei.noaa.gov/pub/data/"
    "swdi/stormevents/csvfiles/"
)

response = requests.get(BASE_URL, timeout=60)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

files = [
    a.get("href")
    for a in soup.find_all("a", href=True)
    if a.get("href", "").startswith("StormEvents_details")
    and a.get("href", "").endswith(".csv.gz")
]

all_wildfires = []

for year in range(2000, 2027):

    matching = [
        f for f in files
        if f"_d{year}_" in f
    ]

    if not matching:
        print(f"No file found for {year}")
        continue

    filename = sorted(matching)[-1]

    print(f"Downloading {year}...")

    r = requests.get(BASE_URL + filename, timeout=180)
    r.raise_for_status()

    yearly_df = pd.read_csv(
        BytesIO(r.content),
        compression="gzip",
        low_memory=False
    )

    yearly_df = yearly_df[
        yearly_df["EVENT_TYPE"].eq("Wildfire")
    ].copy()

    all_wildfires.append(yearly_df)

wildfire_df = pd.concat(
    all_wildfires,
    ignore_index=True
)

print("Total wildfire records:", len(wildfire_df))

Total wildfire records: 9259


In [8]:
import numpy as np

def parse_damage(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper().replace(",", "")

    if value in ["", "NAN", "NONE", "NULL"]:
        return np.nan

    multipliers = {
        "K": 1_000,
        "M": 1_000_000,
        "B": 1_000_000_000
    }

    suffix = value[-1]

    if suffix in multipliers:
        number = value[:-1]

        if not number:
            return np.nan

        return pd.to_numeric(
            number, errors="coerce"
        ) * multipliers[suffix]

    return pd.to_numeric(value, errors="coerce")


# Convert damage columns
for col in ["DAMAGE_PROPERTY", "DAMAGE_CROPS"]:
    wildfire_df[col + "_NUM"] = (
        wildfire_df[col].apply(parse_damage)
    )

# Total economic damage
wildfire_df["total_damage_usd"] = wildfire_df[
    ["DAMAGE_PROPERTY_NUM", "DAMAGE_CROPS_NUM"]
].sum(axis=1, min_count=1)

# Convert fatalities and injuries
casualty_columns = [
    "DEATHS_DIRECT",
    "DEATHS_INDIRECT",
    "INJURIES_DIRECT",
    "INJURIES_INDIRECT"
]

for col in casualty_columns:
    wildfire_df[col] = pd.to_numeric(
        wildfire_df[col],
        errors="coerce"
    ).fillna(0)

wildfire_df["total_deaths"] = (
    wildfire_df["DEATHS_DIRECT"]
    + wildfire_df["DEATHS_INDIRECT"]
)

wildfire_df["total_injuries"] = (
    wildfire_df["INJURIES_DIRECT"]
    + wildfire_df["INJURIES_INDIRECT"]
)

In [9]:
major_wildfires = wildfire_df[
    (wildfire_df["total_damage_usd"] >= 100_000_000)
    | (wildfire_df["total_deaths"] >= 10)
    | (wildfire_df["total_injuries"] >= 50)
].copy()

major_wildfires = major_wildfires.sort_values(
    "total_damage_usd",
    ascending=False,
    na_position="last"
)

print("Major wildfire records:", len(major_wildfires))

Major wildfire records: 44


In [10]:
wildfire_columns = [
    "EVENT_ID",
    "EPISODE_ID",
    "BEGIN_DATE_TIME",
    "END_DATE_TIME",
    "STATE",
    "CZ_NAME",
    "BEGIN_LOCATION",
    "total_damage_usd",
    "total_deaths",
    "total_injuries",
    "BEGIN_LAT",
    "BEGIN_LON",
    "EVENT_NARRATIVE"
]

major_wildfires = major_wildfires[
    wildfire_columns
].copy()

major_wildfires = major_wildfires.rename(columns={
    "EVENT_ID": "wildfire_id",
    "EPISODE_ID": "episode_id",
    "BEGIN_DATE_TIME": "start_date",
    "END_DATE_TIME": "end_date",
    "STATE": "state",
    "CZ_NAME": "county",
    "BEGIN_LOCATION": "location",
    "BEGIN_LAT": "start_lat",
    "BEGIN_LON": "start_lon",
    "EVENT_NARRATIVE": "event_description"
})

# Convert dates
major_wildfires["start_date"] = pd.to_datetime(
    major_wildfires["start_date"],
    errors="coerce"
)

major_wildfires["end_date"] = pd.to_datetime(
    major_wildfires["end_date"],
    errors="coerce"
)

# Calculate duration
major_wildfires["duration_days"] = (
    major_wildfires["end_date"]
    - major_wildfires["start_date"]
).dt.total_seconds() / 86400

print(major_wildfires.head())

      wildfire_id  episode_id          start_date            end_date  \
6156       788675      131864 2018-11-08 06:33:00 2018-11-25 15:00:00   
8129      1118193      183072 2023-08-08 06:40:00 2023-08-16 00:00:00   
7411       992562      164114 2021-12-30 11:15:00 2021-12-30 23:00:00   
205       5144511     1095102 2000-05-04 23:00:00 2000-05-31 23:59:00   
6434       774715      129303 2018-07-23 15:00:00 2018-07-31 23:59:00   

           state                                             county  \
6156  CALIFORNIA              NORTHEAST FOOTHILLS/SACRAMENTO VALLEY   
8129      HAWAII                                  MAUI LEEWARD WEST   
7411    COLORADO  BOULDER & JEFFERSON COUNTIES BELOW 6000 FEET /...   
205   NEW MEXICO                                         LOS ALAMOS   
6434  CALIFORNIA                         NORTHERN SACRAMENTO VALLEY   

        location  total_damage_usd  total_deaths  total_injuries  start_lat  \
6156         NaN      1.700000e+10            86       

/var/folders/88/372_s7qj62b3qk7xqrcjxflw0000gn/T/ipykernel_21365/4194358895.py:35: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  major_wildfires["start_date"] = pd.to_datetime(
/var/folders/88/372_s7qj62b3qk7xqrcjxflw0000gn/T/ipykernel_21365/4194358895.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  major_wildfires["end_date"] = pd.to_datetime(


In [11]:
major_wildfires.to_csv(
    "major_wildfires_2000_2026.csv",
    index=False
)